[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mnrozhkov/santa-scale-challenge/blob/serverless/notebooks/04_video_generation.ipynb)

# Video generation (card → clip → music → MP4)

End-to-end playground: **llm + image** (`make_card`) → **video** (Wan) → **audio**
(ACE-Step) → **mux** → one MP4. Same adapters the CLI uses (`santa card` / `santa animate`).

**Fallback** (`SANTA_FALLBACK`, default `auto`) applies **per role**. Image → OpenAI
`gpt-image-1`; video → `sora-2`; audio → bundled `local_tracks`; llm → `gpt-4o-mini`.
`off` requires every primary endpoint; `only` runs fallbacks without waiting for yours.

**Secrets.** Locally `.env`. On Colab, userdata names match `.env.example`:
`TOKEN_FACTORY_API_KEY`, `IMAGE_ENDPOINT_*`, `VIDEO_ENDPOINT_*`, `AUDIO_ENDPOINT_*`,
and/or `OPENAI_API_KEY`.


In [ ]:
from pathlib import Path
import os
import subprocess
import sys


def apply_colab_secrets() -> None:
    """Copy Colab userdata into os.environ. Secret names match .env.example."""
    try:
        from google.colab import userdata
    except ImportError:
        return
    for key in (
        "IMAGE_ENDPOINT_URL",
        "IMAGE_ENDPOINT_TOKEN",
        "VIDEO_ENDPOINT_URL",
        "VIDEO_ENDPOINT_TOKEN",
        "AUDIO_ENDPOINT_URL",
        "AUDIO_ENDPOINT_TOKEN",
        "TOKEN_FACTORY_API_KEY",
        "OPENAI_API_KEY",
        "SANTA_FALLBACK",
    ):
        try:
            value = userdata.get(key)
        except Exception:
            continue
        if value:
            os.environ[key] = str(value)


try:
    import google.colab  # noqa: F401
except ImportError:
    pass
else:
    apply_colab_secrets()
    repo = Path("/content/santa-scale-challenge")
    if not (repo / "pyproject.toml").is_file():
        subprocess.check_call(
            [
                "git",
                "clone",
                "--branch",
                "serverless",
                "--depth",
                "1",
                "https://github.com/mnrozhkov/santa-scale-challenge.git",
                str(repo),
            ]
        )
    os.chdir(repo)
    if str(repo) not in sys.path:
        sys.path.insert(0, str(repo))
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-e", str(repo)])

from santa.config import Settings

settings = Settings.load()
print("SANTA_FALLBACK =", os.environ.get("SANTA_FALLBACK", "auto"))
print(
    "env present:",
    [
        k
        for k in (
            "IMAGE_ENDPOINT_URL",
            "IMAGE_ENDPOINT_TOKEN",
            "VIDEO_ENDPOINT_URL",
            "VIDEO_ENDPOINT_TOKEN",
            "AUDIO_ENDPOINT_URL",
            "AUDIO_ENDPOINT_TOKEN",
            "TOKEN_FACTORY_API_KEY",
            "OPENAI_API_KEY",
        )
        if os.environ.get(k)
    ],
)


## Parameters

- **kid** — `name`, `age` (1–18), `wish` for `make_card` (llm + Sana).
- **`SEED`** — image seed; video seed is separate (`VIDEO_SEED`).
- **motion prompt** — `MOTION` or `config/prompts.yaml`.
- **mood** — from the wish (`santa.prompts.MOODS`); drives ACE-Step / local tracks.
- **size / num_frames / fps** — `roles.video.options` on `Settings`.
- **`audio_duration`** — `roles.audio.options` in yaml (`Settings`), not a function arg.

Skip `make_card` if you already have a PNG: set `png = Path("…/card.png").read_bytes()`
and a `mood` string, then run the Wan / ACE-Step / mux cell.


In [ ]:
from IPython.display import Image, display
from santa.card import make_card, validate_profile

NAME = "Mia"
AGE = 8
WISH = "a glowing telescope"
SEED = 7  # image seed; or None

kid = validate_profile(NAME, AGE, WISH)
run = make_card(kid, settings, out_root=Path("out"), seed=SEED)
png = Path(run.card.png_path).read_bytes()
mood = run.card.wish.mood

print(run.card.wish.text)
print("mood=", mood, "png=", run.card.png_path)
for step in run.steps:
    print(step)
display(Image(data=png, format="png"))


In [ ]:
from IPython.display import Audio, Video, display
from santa.animate import load_prompts, motion_prompt, music_prompt
from santa.models import adapter_for
from santa.mux import mux

MOTION = None
VIDEO_SEED = None

prompts = load_prompts()
motion = motion_prompt(prompts, MOTION)
video = adapter_for("video", settings)
audio = adapter_for("audio", settings)
# video.cfg.options["size"] = "832x480"
# audio.cfg.options["audio_duration"] = 8

mp4 = video.generate(motion, image=png, seed=VIDEO_SEED)
mp3 = audio.generate(music_prompt(prompts, mood), mood=mood)

out = Path("out") / kid.id
clip = out / "clip.mp4"
track = out / "track.mp3"
dest = out / "card.mp4"
clip.write_bytes(mp4)
track.write_bytes(mp3)
mux(clip, track, dest)

print("video", video.cfg.label, "fallback=", video.used_fallback)
print("audio", audio.cfg.label, "fallback=", audio.used_fallback)
print("muxed", dest)
display(Audio(data=mp3, autoplay=False))
display(Video(str(dest), embed=True, width=640))
